In [1]:
import joblib
import shap
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


In [2]:
X_train_scaled = pd.read_csv('../Data/Sampled_Data/X_train_scaled.csv')
X_test_scaled = pd.read_csv('../Data/Sampled_Data/X_test_scaled.csv')

X_train = pd.read_csv('../Data/Sampled_Data/X_train.csv')
X_test = pd.read_csv('../Data/Sampled_Data/X_test.csv')

y_train = pd.read_csv('../Data/Sampled_Data/y_train.csv').squeeze()
y_test = pd.read_csv('../Data/Sampled_Data/y_test.csv').squeeze()


print("Train shape:", X_train_scaled.shape)
print("Test shape:", X_test_scaled.shape)

y_train_bin = (y_train != 0).astype(int)
print(pd.Series(y_train_bin).value_counts())

y_test_bin = (y_test != 0).astype(int)
print(pd.Series(y_test_bin).value_counts())


Train shape: (2262300, 78)
Test shape: (565576, 78)
0
0    1817055
1     445245
Name: count, dtype: int64
0
0    454265
1    111311
Name: count, dtype: int64


In [3]:
rf = joblib.load('../Data/Sampled_Data/rf.pkl')

In [4]:
from sklearn.model_selection import train_test_split

X_sample, _, y_sample, _ = train_test_split(
    X_test,
    y_test,
    test_size=0.9,
    stratify=y_test,
    random_state=42
)


In [5]:
print(type(X_sample))


<class 'pandas.core.frame.DataFrame'>


In [6]:
from sklearn.utils import resample

X_sample_shap = resample(X_sample, n_samples=1500, random_state=42)

X_sample_shap = pd.DataFrame(
    X_sample_shap,
    columns=X_sample.columns
)


In [7]:
explainer = shap.TreeExplainer(rf)
shap_values = explainer.shap_values(X_sample_shap)

In [8]:
print("Original SHAP shape:", shap_values.shape)

if len(shap_values.shape) == 3:
    shap_values = shap_values[:, :, 1]

print("After class selection:", shap_values.shape)

mean_abs = np.mean(np.abs(shap_values), axis=0)

rf_importance = pd.DataFrame({
    "feature": X_sample_shap.columns,
    "importance": mean_abs
}).sort_values("importance", ascending=False)

Original SHAP shape: (1500, 78, 2)
After class selection: (1500, 78)


In [9]:
print(rf_importance.head(10))

   feature  importance
0        0    0.044086
5        5    0.034974
13      13    0.029874
65      65    0.025538
12      12    0.025083
52      52    0.024968
67      67    0.023184
42      42    0.022502
41      41    0.021803
54      54    0.021191


In [10]:
overall_magnitude = np.mean(np.abs(shap_values))
print("RF baseline SHAP magnitude:", overall_magnitude)

RF baseline SHAP magnitude: 0.007110072405893693


In [11]:
variance = np.var(np.abs(shap_values), axis=0)

rf_variance = pd.DataFrame({
    "feature": X_sample.columns,
    "variance": variance
}).sort_values("variance", ascending=False)

In [12]:
rf_importance.to_csv("../Data/Sampled_Data/rf_baseline_shap.csv", index=False)
rf_variance.to_csv("../Data/Sampled_Data/rf_baseline_variance.csv", index=False)
del rf_importance
del rf_variance
del rf

In [13]:
rf_smote = joblib.load("../Data/Sampled_Data/rf_smote.pkl")

In [14]:
explainer = shap.TreeExplainer(rf_smote)
shap_values = explainer.shap_values(X_sample_shap)

In [15]:
print("Original SHAP shape:", shap_values.shape)

if len(shap_values.shape) == 3:
    shap_values = shap_values[:, :, 1]

print("After class selection:", shap_values.shape)

mean_abs = np.mean(np.abs(shap_values), axis=0)

rf_smote_importance = pd.DataFrame({
    "feature": X_sample_shap.columns,
    "importance": mean_abs
}).sort_values("importance", ascending=False)

Original SHAP shape: (1500, 78, 2)
After class selection: (1500, 78)


In [16]:
print(rf_smote_importance.head(10))

   feature  importance
0        0    0.044086
5        5    0.034974
13      13    0.029874
65      65    0.025538
12      12    0.025083
52      52    0.024968
67      67    0.023184
42      42    0.022502
41      41    0.021803
54      54    0.021191


In [17]:
overall_magnitude = np.mean(np.abs(shap_values))
print("RF baseline SHAP magnitude:", overall_magnitude)

RF baseline SHAP magnitude: 0.007110072405893693


In [18]:
variance = np.var(np.abs(shap_values), axis=0)

rf_smote_variance = pd.DataFrame({
    "feature": X_sample.columns,
    "variance": variance
}).sort_values("variance", ascending=False)

In [19]:
rf_smote_importance.to_csv("../Data/Sampled_Data/rf_smote_shap.csv", index=False)
rf_smote_variance.to_csv("../Data/Sampled_Data/rf_smote_variance.csv", index=False)
del rf_smote_importance
del rf_smote_variance
del rf_smote

In [20]:
rf_smote_bl = joblib.load("../Data/Sampled_Data/rf_smote_bl.pkl")

In [21]:
explainer = shap.TreeExplainer(rf_smote_bl)
shap_values = explainer.shap_values(X_sample_shap)

In [22]:
print("Original SHAP shape:", shap_values.shape)

if len(shap_values.shape) == 3:
    shap_values = shap_values[:, :, 1]

print("After class selection:", shap_values.shape)  

mean_abs = np.mean(np.abs(shap_values), axis=0)
rf_smote_bl_importance = pd.DataFrame({
    "feature": X_sample_shap.columns,
    "importance": mean_abs
}).sort_values("importance", ascending=False)


Original SHAP shape: (1500, 78, 2)
After class selection: (1500, 78)


In [23]:
print(rf_smote_bl_importance.head(10))

   feature  importance
0        0    0.044086
5        5    0.034974
13      13    0.029874
65      65    0.025538
12      12    0.025083
52      52    0.024968
67      67    0.023184
42      42    0.022502
41      41    0.021803
54      54    0.021191


In [24]:
variance = np.var(np.abs(shap_values), axis=0)
rf_smote_bl_variance = pd.DataFrame({
    "feature": X_sample.columns,
    "variance": variance
}).sort_values("variance", ascending=False)


In [25]:
rf_smote_bl_importance.to_csv("../Data/Sampled_Data/rf_smote_bl_shap.csv", index=False)
rf_smote_bl_variance.to_csv("../Data/Sampled_Data/rf_smote_bl_variance.csv", index=False)
del rf_smote_bl_importance
del rf_smote_bl_variance
del rf_smote_bl

In [ ]:
XG_smote = joblib.load("../Data/Sampled_Data/XG_smote.pkl")

In [27]:
explainer = shap.TreeExplainer(XG_smote)
shap_values = explainer.shap_values(X_sample_shap)

In [28]:
print("Original SHAP shape:", shap_values.shape)

if len(shap_values.shape) == 3:
    shap_values = shap_values[:, :, 1]

print("After class selection:", shap_values.shape)

mean_abs = np.mean(np.abs(shap_values), axis=0)

XG_smote_importance = pd.DataFrame({
    "feature": X_sample_shap.columns,
    "importance": mean_abs
}).sort_values("importance", ascending=False)

Original SHAP shape: (1500, 78)
After class selection: (1500, 78)


In [30]:
print(XG_smote_importance.head(10))
overall_magnitude = np.mean(np.abs(shap_values))
print("XG smote SHAP magnitude:", overall_magnitude)

   feature  importance
0        0    1.980217
66      66    1.980047
67      67    1.429345
11      11    1.340797
12      12    0.759722
13      13    0.691392
69      69    0.549542
34      34    0.505533
19      19    0.463723
37      37    0.424489
XG smote SHAP magnitude: 0.21154392


In [31]:
variance = np.var(np.abs(shap_values), axis=0)
XG_smote_variance = pd.DataFrame({
    "feature": X_sample.columns,
    "variance": variance
}).sort_values("variance", ascending=False)

In [32]:
XG_smote_importance.to_csv("../Data/Sampled_Data/XG_smote_importance.csv", index=False)
XG_smote_variance.to_csv("../Data/Sampled_Data/XG_smote_variance.csv", index=False)
del XG_smote_importance
del XG_smote_variance
del XG_smote

In [34]:
XG_smote_BL = joblib.load("../Data/Sampled_Data/XG_smote_BL.pkl")

In [37]:
explainer = shap.TreeExplainer(XG_smote_BL)
shap_values = explainer.shap_values(X_sample_shap)

In [38]:
print("Original SHAP shape:", shap_values.shape)

if len(shap_values.shape) == 3:
    shap_values = shap_values[:, :, 1]

print("After class selection:", shap_values.shape)

mean_abs = np.mean(np.abs(shap_values), axis=0)

XG_smote_BL_importance = pd.DataFrame({
    "feature": X_sample_shap.columns,
    "importance": mean_abs
}).sort_values("importance", ascending=False)

Original SHAP shape: (1500, 78)
After class selection: (1500, 78)


In [39]:
print(XG_smote_BL_importance.head(10))
overall_magnitude = np.mean(np.abs(shap_values))
print("XG smote BL SHAP magnitude:", overall_magnitude)

   feature  importance
66      66    2.152391
0        0    1.869155
11      11    1.358727
67      67    1.302210
6        6    0.893309
69      69    0.822965
13      13    0.567731
42      42    0.391350
12      12    0.374734
19      19    0.357365
XG smote BL SHAP magnitude: 0.20722918


In [40]:
variance = np.var(np.abs(shap_values), axis=0)
XG_smote_BL_variance = pd.DataFrame({
    "feature": X_sample_shap.columns,
    "variance": variance
}).sort_values("variance", ascending=False)

In [41]:
XG_smote_BL_importance.to_csv("../Data/Sampled_Data/XG_smote_BL_importance.csv", index=False)
XG_smote_BL_variance.to_csv("../Data/Sampled_Data/XG_smote_BL_variance.csv", index=False)
del XG_smote_BL_importance
del XG_smote_BL_variance
del XG_smote_BL

In [42]:
XG_smote_kmeans = joblib.load("../Data/Sampled_Data/XG_smote_kmeans.pkl")


In [43]:
explainer = shap.TreeExplainer(XG_smote_kmeans)
shap_values = explainer.shap_values(X_sample_shap)

In [44]:
print("Original SHAP shape:", shap_values.shape)

if len(shap_values.shape) == 3:
    shap_values = shap_values[:, :, 1]

print("After class selection:", shap_values.shape)

mean_abs = np.mean(np.abs(shap_values), axis=0)

XG_smote_kmeans_importance = pd.DataFrame({
    "feature": X_sample_shap.columns,
    "importance": mean_abs
}).sort_values("importance", ascending=False)

Original SHAP shape: (1500, 78)
After class selection: (1500, 78)


In [45]:
print(XG_smote_kmeans_importance.head(10))
overall_magnitude = np.mean(np.abs(shap_values))
print("XG smote kmeans SHAP magnitude:", overall_magnitude)

   feature  importance
0        0    1.910491
66      66    1.902526
67      67    1.203901
11      11    1.107965
13      13    0.759613
46      46    0.617282
52      52    0.586365
12      12    0.567251
69      69    0.456821
34      34    0.376057
XG smote kmeans SHAP magnitude: 0.1971589


In [48]:
variance = np.var(np.abs(shap_values), axis=0)
XG_smote_kmeans_variance = pd.DataFrame({
    "feature": X_sample_shap.columns,
    "variance": variance
}).sort_values("variance", ascending=False)

In [50]:
XG_smote_kmeans_importance.to_csv("../Data/Sampled_Data/XG_smote_kmeans_importance.csv", index=False)
XG_smote_kmeans_variance.to_csv("../Data/Sampled_Data/XG_smote_kmeans_variance.csv", index=False)
del XG_smote_kmeans_importance
del XG_smote_kmeans_variance
del XG_smote_kmeans